# Baseline predictions - re-create the base paper DTF forecasts (Google Colab)

Re-runs **exactly the same baseline code** as the main notebook (one LightGBM per store, Optuna TPE, 21 trials, same seed)
and saves its **daily test predictions**, which were not kept the first time. They are needed for the official
M5 multi-level evaluation (base paper Table 4). It also checks that the result matches your saved `baseline_dtf_results.csv`.

**How to run:** Runtime -> Disconnect and delete runtime, then Runtime -> Run all (about 60-80 minutes). Keep the tab open.
At the end `baseline_preds.zip` downloads.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q lightgbm optuna cmaes shap pyarrow


In [ ]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler, CmaEsSampler
from optuna.pruners import MedianPruner, NopPruner
import warnings, gc, glob, shutil
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "/content/drive/MyDrive/m5_data"
OUT_DIR = "/content/drive/MyDrive/m5_results"
if not os.path.exists(f"{DATA_DIR}/sell_prices.csv"):
    raise FileNotFoundError("Put the M5 CSV files in My Drive/m5_data")
print("Files:", sorted(os.listdir(DATA_DIR)))

WORK_DIR = "/content/m5_work"            # big working files stay on Colab disk (fast, not lost by Drive sync)
MERGED_DIR = f"{WORK_DIR}/merged_by_store"
FEATS_STORE_DIR = f"{WORK_DIR}/features"
for d in [OUT_DIR, MERGED_DIR, FEATS_STORE_DIR]:
    os.makedirs(d, exist_ok=True)

store_list = ["CA_1", "TX_1", "WI_1"]    # one store per state

FORECAST_HORIZON = 7
TEST_DAYS = 90
TRAIN_WINDOW_DAYS = 365
VALID_DAYS = 56
N_TRIALS_BASELINE = 21                  # base paper's number of trials
N_TRIALS_OPTIMIZED = 40
N_TRIALS_JOINT_PER_STRATEGY = 20
PRUNE_CHECK_EVERY = 25
RANDOM_SEED = 42


## Step 1 - Data preparation (one store at a time, saved to disk)

In [ ]:
cal = pd.read_csv(f"{DATA_DIR}/calendar.csv", parse_dates=["date"])
if "d" not in cal.columns:
    cal["d"] = ["d_" + str(i + 1) for i in range(len(cal))]
prices = pd.read_csv(f"{DATA_DIR}/sell_prices.csv")

sales_path = f"{DATA_DIR}/sales_train_evaluation.csv"
id_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in pd.read_csv(sales_path, nrows=0).columns if c.startswith("d_")]
print("Stores used:", store_list)

snap_cols = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
for store_id in store_list:
    out_path = f"{MERGED_DIR}/{store_id}.parquet"
    if os.path.exists(out_path):
        print(f"{store_id}: already prepared, skipping")
        continue
    parts = [ch[ch["store_id"] == store_id] for ch in pd.read_csv(sales_path, chunksize=2000)]
    sales_store = pd.concat(parts, ignore_index=True); del parts
    state = sales_store["state_id"].iloc[0]
    long_store = sales_store.melt(id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="sales")
    long_store["sales"] = long_store["sales"].astype("float32")
    del sales_store; gc.collect()
    long_store = long_store.merge(
        cal[["d", "date", "wm_yr_wk", "wday", "event_name_1", "event_type_1", snap_cols[state]]],
        on="d", how="left").rename(columns={snap_cols[state]: "snap"})
    long_store = long_store.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    long_store.to_parquet(out_path, index=False)
    print(f"{store_id}: saved {len(long_store):,} rows")
    del long_store; gc.collect()

max_date = cal["date"].max()
test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)
print(f"Test period: {test_start.date()} to {max_date.date()}")


## Step 2 - Feature engineering (base paper Appendix A, adapted to M5) and shared functions

In [ ]:
LAGS = list(range(1, 15))
WINDOWS = [7, 14, 30, 60, 180]
MANUAL_FEATURES = (
    [f"sales_lag_{l}" for l in LAGS]
    + [f"rolling_mean_{w}" for w in WINDOWS] + [f"rolling_stdv_{w}" for w in WINDOWS]
    + ["day_of_week", "day_of_month", "week_of_year", "month", "quarter", "end_of_week",
       "sell_price", "price_ratio_to_mean", "price_ratio_to_max", "snap", "is_event"]
)
CAT_FEATURES = ["dept_id", "cat_id"]
KEY = ["store_id", "item_id"]

def build_features(df, horizon=FORECAST_HORIZON):
    """Vectorised version (much faster and lighter than a per-item loop)."""
    df = df.sort_values(["item_id", "date"]).reset_index(drop=True)
    g = df.groupby("item_id", sort=False)["sales"]
    for l in LAGS:
        df[f"sales_lag_{l}"] = g.shift(l).astype("float32")
    df["_shifted"] = g.shift(horizon)
    gs = df.groupby("item_id", sort=False)["_shifted"]
    for w in WINDOWS:
        df[f"rolling_mean_{w}"] = gs.rolling(w).mean().reset_index(level=0, drop=True).astype("float32")
        df[f"rolling_stdv_{w}"] = gs.rolling(w).std().reset_index(level=0, drop=True).astype("float32")
    df = df.drop(columns="_shifted")
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["end_of_week"] = (df["day_of_week"] >= 5).astype(int)
    df["is_event"] = df["event_name_1"].notna().astype(int)
    gp = df.groupby("item_id", sort=False)["sell_price"]
    csum = df["sell_price"].fillna(0).groupby(df["item_id"], sort=False).cumsum()
    ccnt = df["sell_price"].notna().groupby(df["item_id"], sort=False).cumsum()
    df["price_ratio_to_mean"] = (df["sell_price"] / (csum / ccnt.replace(0, np.nan))).astype("float32")
    df["price_ratio_to_max"] = (df["sell_price"] / gp.cummax()).astype("float32")
    return df

def to_category(df):
    df = df.copy()
    for c in CAT_FEATURES:
        df[c] = df[c].astype("category")
    return df

def rmsse(y_true, y_pred, hist):
    """Base paper Eq. 1"""
    denom = np.mean(np.diff(hist) ** 2) if len(hist) > 1 else 0
    return np.sqrt(np.mean((y_true - y_pred) ** 2) / (denom if denom > 0 else 1e-9))

def sme(y_true, y_pred, hist):
    """Base paper Eq. 2"""
    denom = np.mean(np.abs(np.diff(hist))) if len(hist) > 1 else 0
    return np.mean(y_true - y_pred) / (denom if denom > 0 else 1e-9)

def suggest_params(trial):
    return {
        "objective": "tweedie",
        "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.01, 1.9),
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 0.8),
        "max_bin": trial.suggest_int("max_bin", 100, 150),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 1.5),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 1.5),
    }

FIXED_PARAMS = {"subsample": 0.5, "subsample_freq": 1, "verbosity": -1,
                "seed": RANDOM_SEED, "n_jobs": -1}

def split_train_valid(df):
    """Same rule for every method: the last VALID_DAYS of the training data are used for tuning."""
    cut = df["date"].max() - pd.Timedelta(days=VALID_DAYS - 1)
    return df[df["date"] < cut], df[df["date"] >= cut]

def pruning_callback(trial):
    """Reports validation RMSE to Optuna every PRUNE_CHECK_EVERY rounds so bad trials stop early."""
    def _cb(env):
        if (env.iteration + 1) % PRUNE_CHECK_EVERY:
            return
        for item in env.evaluation_result_list:
            if item[1] == "rmse":
                trial.report(item[2], env.iteration)
                if trial.should_prune():
                    raise optuna.TrialPruned()
    return _cb

def fit_lgb(params, tr, va=None, trial=None):
    model = lgb.LGBMRegressor(**params, **FIXED_PARAMS)
    X_tr = tr[MANUAL_FEATURES + CAT_FEATURES]
    if trial is not None and va is not None:
        model.fit(X_tr, tr["sales"], categorical_feature=CAT_FEATURES,
                  eval_set=[(va[MANUAL_FEATURES + CAT_FEATURES], va["sales"])],
                  eval_metric="rmse", callbacks=[pruning_callback(trial)])
    else:
        model.fit(X_tr, tr["sales"], categorical_feature=CAT_FEATURES)
    return model

def tune_and_train(df, n_trials, sampler, use_pruning=False):
    """Tune on (train, valid) split, then refit the best configuration on all training data."""
    df = to_category(df.dropna(subset=MANUAL_FEATURES + ["sales"]))
    tr, va = split_train_valid(df)
    pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=PRUNE_CHECK_EVERY) if use_pruning else NopPruner()

    def objective(trial):
        params = suggest_params(trial)
        model = fit_lgb(params, tr, va, trial if use_pruning else None)
        pred = model.predict(va[MANUAL_FEATURES + CAT_FEATURES])
        del model; gc.collect()
        return float(np.sqrt(np.mean((va["sales"].values - pred) ** 2)))

    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
    final_model = fit_lgb(study.best_params, df)
    del tr, va; gc.collect()
    return final_model, study.best_value, study.best_params, n_pruned

PREDS = []   # test-period predictions of every method (saved at the end for analysis / dashboard)

def evaluate_on_test(model, test_df, train_df, method=None):
    """Per product-store RMSSE and SME on the test period (base paper Eq. 1 and 2)."""
    test_df = to_category(test_df.dropna(subset=MANUAL_FEATURES))
    test_df = test_df.assign(pred=model.predict(test_df[MANUAL_FEATURES + CAT_FEATURES]))
    if method:
        PREDS.append(test_df[KEY + ["date", "sales", "pred"]].assign(method=method))
    hist = {k: g["sales"].values for k, g in train_df.sort_values("date").groupby(KEY)}
    rows = []
    for k, g in test_df.groupby(KEY):
        h = hist.get(k)
        if h is None or len(h) < 2:
            continue
        rows.append({"store_id": k[0], "item_id": k[1],
                     "rmsse": rmsse(g["sales"].values, g["pred"].values, h),
                     "sme": sme(g["sales"].values, g["pred"].values, h),
                     "max_pred": float(g["pred"].max()), "max_actual": float(g["sales"].max())})
    return pd.DataFrame(rows)

def result_path(name):
    return f"{OUT_DIR}/{name}.csv"

def log_tuning(row):
    """Append one tuning record to Drive immediately (survives disconnects)."""
    path = result_path("tuning_log")
    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

print("Functions ready. Features used:", len(MANUAL_FEATURES) + len(CAT_FEATURES))


## Step 3 - Build and cache features for each store (done once, reused by every method)

In [ ]:
def feature_paths(store_id):
    return f"{FEATS_STORE_DIR}/{store_id}_train.parquet", f"{FEATS_STORE_DIR}/{store_id}_test.parquet"

for store_id in store_list:
    tr_path, te_path = feature_paths(store_id)
    if os.path.exists(tr_path) and os.path.exists(te_path):
        print(f"{store_id}: features already built, skipping")
        continue
    raw = pd.read_parquet(f"{MERGED_DIR}/{store_id}.parquet")
    # keep TRAIN_WINDOW_DAYS of training data + 200 days so the 180-day rolling features can be computed
    raw = raw[raw["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS + 200)]
    feats = build_features(raw); del raw; gc.collect()
    train = feats[(feats["date"] < test_start) & (feats["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS))]
    test = feats[feats["date"] >= test_start]
    train.to_parquet(tr_path, index=False)
    test.to_parquet(te_path, index=False)
    print(f"{store_id}: train {len(train):,} rows, test {len(test):,} rows")
    del feats, train, test; gc.collect()

def load_store(store_id):
    tr_path, te_path = feature_paths(store_id)
    return pd.read_parquet(tr_path), pd.read_parquet(te_path)


## Step 4 - Baseline: base paper DTF (re-run, predictions saved)

In [ ]:
baseline_results = []
for store_id in store_list:
    tr, te = load_store(store_id)
    model, val, params, _ = tune_and_train(tr, N_TRIALS_BASELINE, TPESampler(seed=RANDOM_SEED))
    res = evaluate_on_test(model, te, tr, method="Base paper DTF (replicated)")
    baseline_results.append(res)
    pd.DataFrame([{"method": "Baseline DTF (rerun)", "group": store_id, "valid_rmse": val, **params}]).to_csv(
        f"{OUT_DIR}/baseline_rerun_params.csv", mode="a", header=not os.path.exists(f"{OUT_DIR}/baseline_rerun_params.csv"), index=False)
    print(f"{store_id}: median test RMSSE {res['rmsse'].median():.4f}")
    del tr, te, model; gc.collect()
rerun = pd.concat(baseline_results, ignore_index=True)
rerun.to_csv(f"{OUT_DIR}/baseline_rerun_results.csv", index=False)
pd.concat(PREDS, ignore_index=True).to_csv(f"{OUT_DIR}/baseline_predictions.csv.gz", index=False)
orig_path = result_path("baseline_dtf_results")
if os.path.exists(orig_path):
    m = pd.read_csv(orig_path).merge(rerun, on=KEY, suffixes=("_orig", "_rerun")).dropna(subset=["rmsse_orig", "rmsse_rerun"])
    m = m[m["rmsse_orig"] < 1000]
    print(f"Check vs saved baseline on {len(m):,} products: mean RMSSE {m['rmsse_orig'].mean():.4f} (saved) vs {m['rmsse_rerun'].mean():.4f} (re-run), "
          f"correlation {m['rmsse_orig'].corr(m['rmsse_rerun']):.4f}")


## Step 5 - Download

In [ ]:
import shutil
zdir = f"{OUT_DIR}/_basepred_zip"; os.makedirs(zdir, exist_ok=True)
for f in ["baseline_predictions.csv.gz", "baseline_rerun_results.csv", "baseline_rerun_params.csv"]:
    shutil.copy(f"{OUT_DIR}/{f}", zdir)
shutil.make_archive("/content/baseline_preds", "zip", zdir)
from google.colab import files
files.download("/content/baseline_preds.zip")
